# Claude PE Framework Agent

Prompt Engineering agent interface. Run the setup cell once, then use the chat cell for conversation.

In [ ]:
# Setup - Run once per session
import os
import json
import requests
from pathlib import Path
from IPython.display import display, Markdown

API_URL = "https://api.anthropic.com/v1/messages"
ANTHROPIC_VERSION = "2023-06-01"
MODEL = "claude-sonnet-4-20250514"

# Paths
WORKING_DIR = Path.cwd()
AGENT_FILES = WORKING_DIR / "Agent Files"
HISTORY_DIR = WORKING_DIR / "History"
OUTPUT_DIR = WORKING_DIR / "Output"

# Load system prompt
SYSTEM_PROMPT = (AGENT_FILES / "Instructions.md").read_text()

# Conversation state
conversation_history = []

# Tool definitions
TOOLS = [
    {
        "name": "read_file",
        "description": "Read a file from Agent Files directory",
        "input_schema": {
            "type": "object",
            "properties": {"filename": {"type": "string"}},
            "required": ["filename"]
        }
    },
    {
        "name": "write_file",
        "description": "Write a file to Agent Files directory",
        "input_schema": {
            "type": "object",
            "properties": {"filename": {"type": "string"}, "content": {"type": "string"}},
            "required": ["filename", "content"]
        }
    },
    {
        "name": "list_files",
        "description": "List files in Agent Files directory",
        "input_schema": {"type": "object", "properties": {}}
    },
    {
        "name": "write_output",
        "description": "Write a deliverable to Output directory",
        "input_schema": {
            "type": "object",
            "properties": {"filename": {"type": "string"}, "content": {"type": "string"}},
            "required": ["filename", "content"]
        }
    }
]

def execute_tool(name, inputs):
    """Execute a tool and return the result."""
    if name == "read_file":
        path = AGENT_FILES / inputs["filename"]
        return path.read_text() if path.exists() else f"File not found: {inputs['filename']}"
    elif name == "write_file":
        path = AGENT_FILES / inputs["filename"]
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_text(inputs["content"])
        return f"Wrote {inputs['filename']}"
    elif name == "list_files":
        files = [f.name for f in AGENT_FILES.iterdir() if f.is_file()]
        return "\n".join(sorted(files)) if files else "No files found"
    elif name == "write_output":
        path = OUTPUT_DIR / inputs["filename"]
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_text(inputs["content"])
        return f"Wrote output: {inputs['filename']}"
    return f"Unknown tool: {name}"

def chat(message):
    """Send a message and get a response."""
    global conversation_history
    
    api_key = os.environ.get("ANTHROPIC_API_KEY")
    if not api_key:
        return "Error: ANTHROPIC_API_KEY not set"
    
    conversation_history.append({"role": "user", "content": message})
    
    while True:
        response = requests.post(
            API_URL,
            headers={
                "x-api-key": api_key,
                "anthropic-version": ANTHROPIC_VERSION,
                "content-type": "application/json"
            },
            json={
                "model": MODEL,
                "max_tokens": 4096,
                "system": SYSTEM_PROMPT,
                "messages": conversation_history,
                "tools": TOOLS
            }
        )
        
        if not response.ok:
            return f"API Error: {response.text}"
        
        data = response.json()
        conversation_history.append({"role": "assistant", "content": data["content"]})
        
        # Check for tool use
        tool_uses = [b for b in data["content"] if b["type"] == "tool_use"]
        if not tool_uses:
            # No tools, return text response
            text = "".join(b.get("text", "") for b in data["content"] if b["type"] == "text")
            return text
        
        # Execute tools and continue
        tool_results = []
        for tool in tool_uses:
            print(f"[Tool: {tool['name']}]")
            result = execute_tool(tool["name"], tool["input"])
            tool_results.append({
                "type": "tool_result",
                "tool_use_id": tool["id"],
                "content": result
            })
        
        conversation_history.append({"role": "user", "content": tool_results})

def reset_conversation():
    """Clear conversation history."""
    global conversation_history
    conversation_history = []
    print("Conversation reset.")

print(f"PE Agent ready. Working directory: {WORKING_DIR}")
print(f"Agent Files: {list(f.name for f in AGENT_FILES.iterdir() if f.is_file())}")

In [ ]:
# Chat - Edit message and run
message = """
Hello! What can you help me with?
"""

response = chat(message.strip())
display(Markdown(response))

---
## Agent Files

Use the cells below to view and edit agent files directly.

In [ ]:
# View Agent File - Change filename and run
filename = "Instructions.md"

content = (AGENT_FILES / filename).read_text()
display(Markdown(f"**{filename}**\n\n---\n\n{content}"))

In [ ]:
# Edit Agent File - Modify and run to save
filename = "Instructions.md"
content = """
# Your content here

Edit this cell with the file content, then run to save.
"""

# Uncomment the line below to save:
# (AGENT_FILES / filename).write_text(content.strip())
# print(f"Saved {filename}")

In [ ]:
# List all Agent Files
print("Agent Files:")
for f in sorted(AGENT_FILES.iterdir()):
    if f.is_file():
        print(f"  {f.name}")
    elif f.is_dir():
        print(f"  {f.name}/")
        for sub in sorted(f.iterdir()):
            print(f"    {sub.name}")

---
## Utilities

In [ ]:
# Reset conversation
reset_conversation()

In [ ]:
# Save conversation to History
from datetime import datetime

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
history_file = HISTORY_DIR / f"conversation_{timestamp}.json"
history_file.write_text(json.dumps(conversation_history, indent=2))
print(f"Saved to {history_file}")